# U1 - Integrante 4 - Regresión de Cantidad (qty)

Pipeline Bronze → Silver → Gold con PySpark, entrenamiento y comparación de modelos de regresión (Linear Regression vs Decision Tree Regressor).

## 1. Spark Session

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, LongType, DoubleType, BooleanType
from pyspark.sql.functions import col, when
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
        .appName("U1-Integrante4-RegresionQty")
        .master("local[*]")
        .config("spark.ui.port", "4040")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.memory", "4g")
        .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 15:18:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Carga Bronze

In [2]:
schema = StructType([
    StructField("trade_id", LongType(), False),
    StructField("price", DoubleType(), True),
    StructField("qty", DoubleType(), True),
    StructField("quote_qty", DoubleType(), True),
    StructField("time_us", LongType(), False),
    StructField("is_buyer_maker", BooleanType(), True),
    StructField("is_best_match", BooleanType(), True)
])

df_bronze = spark.read.csv("/opt/data/BTCUSDT-trades-2026-01-05.csv", schema=schema, header=False)
df_bronze.printSchema()
df_bronze.show(5)

root
 |-- trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- qty: double (nullable = true)
 |-- quote_qty: double (nullable = true)
 |-- time_us: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)

+----------+--------+-------+-----------+----------------+--------------+-------------+
|  trade_id|   price|    qty|  quote_qty|         time_us|is_buyer_maker|is_best_match|
+----------+--------+-------+-----------+----------------+--------------+-------------+
|5734054604|91529.74| 2.2E-4| 20.1365428|1767571200308618|         false|         true|
|5734054605|91529.74|   0.01|   915.2974|1767571200375801|         false|         true|
|5734054606|91529.74|0.00437|399.9849638|1767571200477157|         false|         true|
|5734054607|91529.74|0.00764|699.2872136|1767571200480543|         false|         true|
|5734054608|91529.74|0.00136|124.4804464|1767571200545508|         false|         true|
+--------

## 3. Capa Silver

In [3]:
df_silver = df_bronze.na.drop(subset=["trade_id", "qty", "quote_qty"]) \
    .dropDuplicates(["trade_id"]) \
    .withColumn("is_buyer_maker_num", when(col("is_buyer_maker") == True, 1.0).otherwise(0.0))

df_silver.show(5)

[Stage 1:>                                                        (0 + 10) / 10]

+----------+--------+-------+-------------+----------------+--------------+-------------+------------------+
|  trade_id|   price|    qty|    quote_qty|         time_us|is_buyer_maker|is_best_match|is_buyer_maker_num|
+----------+--------+-------+-------------+----------------+--------------+-------------+------------------+
|5734054604|91529.74| 2.2E-4|   20.1365428|1767571200308618|         false|         true|               0.0|
|5734054610|91529.74| 5.4E-4|   49.4260596|1767571200665868|         false|         true|               0.0|
|5734054624|91529.74|0.20378|18651.9304172|1767571202033686|         false|         true|               0.0|
|5734054642|91529.73|0.00599|  548.2630827|1767571202551395|          true|         true|               1.0|
|5734054646|91529.74| 9.0E-5|    8.2376766|1767571202941974|         false|         true|               0.0|
+----------+--------+-------+-------------+----------------+--------------+-------------+------------------+
only showing top 5 

## 4. Capa Gold

In [4]:
df_silver.write.mode("overwrite").parquet("/opt/artifacts/gold_integrante4")
df_gold = spark.read.parquet("/opt/artifacts/gold_integrante4")
df_gold.show(5)

+----------+--------+-------+-----------+----------------+--------------+-------------+------------------+
|  trade_id|   price|    qty|  quote_qty|         time_us|is_buyer_maker|is_best_match|is_buyer_maker_num|
+----------+--------+-------+-----------+----------------+--------------+-------------+------------------+
|5734054606|91529.74|0.00437|399.9849638|1767571200477157|         false|         true|               0.0|
|5734054617|91529.74| 9.6E-4| 87.8685504|1767571201602432|         false|         true|               0.0|
|5734054630|91529.74| 1.1E-4| 10.0682714|1767571202085489|         false|         true|               0.0|
|5734054637|91529.73| 4.7E-4| 43.0189731|1767571202435549|          true|         true|               1.0|
|5734054644|91529.73|0.00476|435.6815148|1767571202744883|          true|         true|               1.0|
+----------+--------+-------+-----------+----------------+--------------+-------------+------------------+
only showing top 5 rows


## 5. Features

In [5]:
assembler = VectorAssembler(inputCols=["price", "quote_qty", "is_buyer_maker_num"], outputCol="features")
df_ml = assembler.transform(df_gold).select("features", col("qty").alias("label"))

df_train, df_test = df_ml.randomSplit([0.8, 0.2], seed=42)

## 6. Modelos

In [6]:
lr = LinearRegression(featuresCol="features", labelCol="label", regParam=0.01)
modelo_lr = lr.fit(df_train)
pred_lr = modelo_lr.transform(df_test)

dt = DecisionTreeRegressor(featuresCol="features", labelCol="label")
modelo_dt = dt.fit(df_train)
pred_dt = modelo_dt.transform(df_test)

netlib-blas: JNI_OnLoad: dlopen(libblas.so.3) failed: libblas.so.3: cannot open shared object file: No such file or directory
netlib-lapack: JNI_OnLoad: dlopen(liblapack.so.3) failed: liblapack.so.3: cannot open shared object file: No such file or directory
                                                                                

## 7. Evaluación (MAE y RMSE)

In [7]:
eval_mae = RegressionEvaluator(metricName="mae")
print(f"Linear Regression MAE: {eval_mae.evaluate(pred_lr):.6f}")
print(f"Decision Tree MAE: {eval_mae.evaluate(pred_dt):.6f}")

Linear Regression MAE: 0.001238


[Stage 32:>                                                         (0 + 8) / 8]

Decision Tree MAE: 0.002761


## 8. Guardar Ganador

In [8]:
modelo_lr.write().overwrite().save("/opt/artifacts/modelo_ganador_integrante4")